In [1]:
# ============================================================
# WEEK 1 — Designing the MDP and Custom Gym Environment
# Project: Travel & Hospitality — RL Dynamic Pricing
# Intern Branch: preeti-dev | Infotact Solutions
# ============================================================
 
 
# ── CELL 1: Install & Import Libraries ───────────────────
import subprocess, sys
 
def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])
 
for lib in ["gymnasium", "numpy", "matplotlib", "seaborn", "pandas"]:
    install(lib)
 
import gymnasium as gym
from gymnasium import spaces
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os
import warnings
warnings.filterwarnings('ignore')
 
os.makedirs('../reports', exist_ok=True)
os.makedirs('../models', exist_ok=True)
os.makedirs('../data',   exist_ok=True)
 
print("✅ All libraries imported successfully")
print(f"   Gymnasium version : {gym.__version__}")
 

✅ All libraries imported successfully
   Gymnasium version : 1.3.0


In [2]:
# ── CELL 2: MDP Problem Formulation ──────────────────────
# Before writing any code, we formally define the MDP.
#
# SCENARIO:
#   An airline has 50 seats on a flight departing in 30 days.
#   Every day it must choose a price. Customers stochastically
#   decide to buy based on that price and urgency.
#
# MDP COMPONENTS:
# ┌─────────────┬────────────────────────────────────────────┐
# │ Component   │ Definition                                 │
# ├─────────────┼────────────────────────────────────────────┤
# │ State (s)   │ [remaining_seats, days_until_departure]    │
# │ Action (a)  │ price_level index → one of 10 price tiers  │
# │ Reward (r)  │ revenue earned that day (price × bookings) │
# │ Transition  │ seats decrease by number of bookings       │
# │ Terminal    │ flight departs (day=0) OR seats sold out   │
# └─────────────┴────────────────────────────────────────────┘
 
print("=" * 55)
print("   MDP FORMULATION — Airline Dynamic Pricing")
print("=" * 55)
print()
print("  State space  : [remaining_seats (0-50), days_left (0-30)]")
print("  Action space : 10 discrete price levels")
print("  Price range  : $50 to $500 (step $50)")
print("  Reward       : price × number of seats booked that day")
print("  Episode end  : day = 0  OR  remaining_seats = 0")
print()
print("  Demand model : P(purchase) = base_prob")
print("                 × price_sensitivity")
print("                 × urgency_factor")
print()
print("✅ MDP formulation complete")
 

   MDP FORMULATION — Airline Dynamic Pricing

  State space  : [remaining_seats (0-50), days_left (0-30)]
  Action space : 10 discrete price levels
  Price range  : $50 to $500 (step $50)
  Reward       : price × number of seats booked that day
  Episode end  : day = 0  OR  remaining_seats = 0

  Demand model : P(purchase) = base_prob
                 × price_sensitivity
                 × urgency_factor

✅ MDP formulation complete


In [4]:
# ── CELL 3: Custom Gym Environment ───────────────────────
class AirlinePricingEnv(gym.Env):
    """
    Custom OpenAI Gymnasium environment for airline seat pricing.
 
    The agent (pricing algorithm) interacts with this environment
    every day — choosing a price, observing how many seats sell,
    and receiving the day's revenue as reward.
 
    State  : [remaining_seats, days_until_departure]
    Action : index into PRICE_LEVELS list
    Reward : price × seats_booked (daily revenue)
    """
 
    metadata = {'render_modes': ['human']}
 
    # Available price tiers (in dollars)
    PRICE_LEVELS = [50, 100, 150, 200, 250, 300, 350, 400, 450, 500]
 
    def __init__(self,
                 total_seats     = 50,
                 total_days      = 30,
                 max_daily_demand= 5,
                 render_mode     = None):
        super().__init__()
 
        self.total_seats      = total_seats
        self.total_days       = total_days
        self.max_daily_demand = max_daily_demand
        self.render_mode      = render_mode
 
        # Action space: choose one of 10 price levels
        self.action_space = spaces.Discrete(len(self.PRICE_LEVELS))
 
        # Observation space: [remaining_seats, days_until_departure]
        self.observation_space = spaces.Box(
            low  = np.array([0, 0],                            dtype=np.float32),
            high = np.array([total_seats, total_days],         dtype=np.float32),
            dtype= np.float32
        )
 
        # Internal state
        self.remaining_seats    = total_seats
        self.days_until_departure = total_days
        self.total_revenue      = 0.0
        self.history            = []   # track each day's events
 
    def _demand_probability(self, price, days_left):
        """
        Stochastic demand function.
 
        Key idea: customers are more likely to buy when:
          1. Price is LOW
          2. Departure is NEAR (last-minute urgency)
          3. There is natural randomness in booking behaviour
 
        Formula breakdown:
          base_prob         = 0.7 (70% max purchase chance)
          price_sensitivity = decreases as price increases
          urgency_factor    = increases as days_left decreases
          noise             = small random fluctuation
        """
        max_price  = max(self.PRICE_LEVELS)   # 500
 
        # Price sensitivity: probability drops as price rises
        # At $50  → factor ≈ 1.0 (very likely to buy)
        # At $500 → factor ≈ 0.0 (very unlikely)
        price_sensitivity = 1.0 - (price / max_price) ** 0.8
 
        # Urgency factor: rises sharply in last 7 days
        # At day 30 → factor = 0.3 (low urgency, early bookers)
        # At day 1  → factor = 1.0 (high urgency, must-book-now)
        urgency_factor = 0.3 + 0.7 * np.exp(-days_left / 7)
 
        # Base probability
        base_prob = 0.7
 
        # Random market noise ± 10%
        noise = np.random.uniform(-0.10, 0.10)
 
        # Final probability — clipped to valid range [0, 1]
        prob = base_prob * price_sensitivity * urgency_factor + noise
        return float(np.clip(prob, 0.0, 1.0))
 
    def _get_obs(self):
        """Return current state as numpy array."""
        return np.array([self.remaining_seats,
                         self.days_until_departure], dtype=np.float32)
 
    def reset(self, seed=None, options=None):
        """Reset environment to start of a new booking season."""
        super().reset(seed=seed)
        self.remaining_seats      = self.total_seats
        self.days_until_departure = self.total_days
        self.total_revenue        = 0.0
        self.history              = []
        return self._get_obs(), {}
 
    def step(self, action):
        """
        Take one step — one day of pricing decisions.
 
        1. Agent chooses a price (action index)
        2. Environment calculates demand probability
        3. Number of bookings is sampled (Poisson-like)
        4. Revenue is calculated and returned as reward
        5. State is updated (seats sold, day passes)
        """
        assert self.action_space.contains(action), f"Invalid action: {action}"
 
        price    = self.PRICE_LEVELS[action]
        days_left= self.days_until_departure
 
        # Compute purchase probability for this price + time
        prob = self._demand_probability(price, days_left)
 
        # Number of customers arriving today (random, max = max_daily_demand)
        arriving_customers = np.random.randint(0, self.max_daily_demand + 1)
 
        # Each customer independently decides to buy
        bookings = sum(np.random.random() < prob
                       for _ in range(arriving_customers))
 
        # Can't sell more than what's available
        bookings = min(bookings, self.remaining_seats)
 
        # Revenue for this day
        daily_revenue = price * bookings
 
        # Update state
        self.remaining_seats      -= bookings
        self.days_until_departure -= 1
        self.total_revenue        += daily_revenue
 
        # Record history
        self.history.append({
            'day'           : self.total_days - days_left + 1,
            'days_left'     : days_left,
            'price'         : price,
            'prob'          : round(prob, 4),
            'arriving'      : arriving_customers,
            'bookings'      : bookings,
            'daily_revenue' : daily_revenue,
            'remaining_seats': self.remaining_seats,
            'total_revenue' : self.total_revenue
        })
 
        # Episode ends when flight departs OR all seats sold
        terminated = (self.days_until_departure == 0 or
                      self.remaining_seats == 0)
        truncated  = False
 
        return self._get_obs(), daily_revenue, terminated, truncated, {}
 
    def render(self):
        """Print current state to console."""
        if self.render_mode == 'human':
            print(f"  Day {self.total_days - self.days_until_departure:>2} | "
                  f"Days left: {self.days_until_departure:>2} | "
                  f"Seats left: {self.remaining_seats:>2} | "
                  f"Revenue: ${self.total_revenue:,.0f}")
 
    def get_history_df(self):
        """Return episode history as a Pandas DataFrame."""
        return pd.DataFrame(self.history)
 
 
print("✅ AirlinePricingEnv class defined")
print()
print("Environment specs:")
print(f"   Seats          : 50")
print(f"   Selling horizon: 30 days")
print(f"   Price levels   : {AirlinePricingEnv.PRICE_LEVELS}")
print(f"   Action space   : Discrete(10)")
print(f"   State space    : Box([seats, days])")
 

✅ AirlinePricingEnv class defined

Environment specs:
   Seats          : 50
   Selling horizon: 30 days
   Price levels   : [50, 100, 150, 200, 250, 300, 350, 400, 450, 500]
   Action space   : Discrete(10)
   State space    : Box([seats, days])


In [ ]:
# ── CELL 4: Validate the Environment ─────────────────────
# Gymnasium provides a built-in checker to verify
# our environment follows the correct API contract
 
from gymnasium.utils.env_checker import check_env
 
env = AirlinePricingEnv()
check_env(env, warn=True)
 
print("✅ Environment passed Gymnasium validation check")